# Fall Detection Pipeline (Nano)

End-to-end real-time pipeline for Jetson Nano:

```
camera/video  ->  MoveNet (TRT FP16)  ->  51-dim features (sliding deque)
              ->  GRU (PyTorch CPU)  ->  fall_prob  ->  alarm logic  ->  display
```

**Run order**: configure (cell 2) -> load models (cell 3) -> define helpers (cell 4)
-> pick a runner cell:
- **cell 5**: live webcam
- **cell 6**: pre-recorded video file
- **cell 7**: quick MoveNet-only sanity check (no GRU)

Stop a running cell with Jupyter's interrupt button (or `q` in the OpenCV window if `DISPLAY=True`).

In [1]:
# === Config ===========================================================
# Paths
ENGINE_PATH = 'movenet/output/pose_fp16.engine'
# GRU_PATH    = 'models/fall_gru_v2.pth'
# GRU_PATH = 'models/fall_gru_v3.pth'
# GRU_PATH = 'models/fall_gru_v4.pth'
# GRU_PATH = 'models/fall_gru_v5.pth'
GRU_PATH = 'models/fall_gru_v6.pth'
# Input
CAMERA_INDEX = 0          # used by the live-webcam cell
VIDEO_PATH   = 'data_Le2i/Coffee_room_01/Coffee_room_01/Videos/video (1).avi'
CAM_WIDTH    = 640
CAM_HEIGHT   = 480

# Alarm
PROB_THRESHOLD = None     # if None, use checkpoint best_threshold or 0.6
ALARM_FRAMES   = 4        # consecutive over-threshold frames to fire (was 6)
ALARM_HOLD     = 50       # frames the alarm stays on after firing

# Rule fusion
RULE_ENABLE       = True
RULE_FORCE_ALARM  = False  # rule alone won't fire; needs GRU agreement (reduces FP)
GRU_MIN_FOR_RULE  = 0.60   # minimum GRU prob to allow rule boost (raised for high-thr model)
RULE_BONUS_PROB   = 0.97   # fused prob floor when rule fires (must be > PROB_THRESHOLD)

RULE_KP_SCORE_THR = 0.20   # min score to trust a keypoint
RULE_DROP_WINDOW  = 5      # frames used to measure rapid drop
RULE_DROP_THR     = 0.75   # normalized drop in torso lengths (was 0.60, tighter)
RULE_ANGLE_THR    = 45.0   # degrees from vertical; >=45 means body is tilted (was 50)
RULE_STILL_WINDOW = 12     # frames of low motion to confirm stillness
RULE_STILL_SPEED  = 0.06   # torso-lengths per frame (was 0.08)
RULE_DROP_HOLD    = 12     # frames to keep a drop event active (was 20)
RULE_MIN_SCALE    = 20.0   # px; ignore tiny people

# Sustained-horizontal rule: fires when body has been tilted for a long window
# AND is still. Catches slow falls / lying on ground without a detected rapid drop.
RULE_SUSTAINED_ANGLE_WINDOW = 20   # consecutive frames body must stay horizontal
RULE_SUSTAINED_ANGLE_THR    = 60.0 # stricter angle threshold for sustained detection

# Inference / display
INFERENCE_STRIDE = 1      # run GRU every N frames; raise if Nano can't keep up
DRAW_KP_THR      = 0.2    # min keypoint score to draw skeleton
SMOOTH_ALPHA     = 0.2    # keypoint EMA smoothing (match training)

DISPLAY = True            # show OpenCV window. Set False for headless / SSH.
SAVE_OUT = None           # e.g. 'demo.mp4' to record annotated output

print('config loaded')


config loaded


In [2]:
# === Imports & model loading ==========================================
import os, sys, time, collections
import cv2
import numpy as np
import torch

# project-root imports
ROOT = os.getcwd()
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, 'movenet'))

from features import KeypointFeatureExtractor, FEATURE_DIM
from fall_model import FallDetectionGRU

# Skeleton edges for visualization (COCO-17)
SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]


def draw_keypoints_fallback(frame, keypoints, conf_thr=0.3,
                            color_pt=(0, 255, 0), color_line=(200, 200, 200)):
    vis = frame.copy()
    for (i, j) in SKELETON:
        if keypoints[i, 2] > conf_thr and keypoints[j, 2] > conf_thr:
            p1 = (int(keypoints[i, 0]), int(keypoints[i, 1]))
            p2 = (int(keypoints[j, 0]), int(keypoints[j, 1]))
            cv2.line(vis, p1, p2, color_line, 2)
    for k in range(17):
        x, y, s = keypoints[k]
        if s > conf_thr:
            cv2.circle(vis, (int(x), int(y)), 4, color_pt, -1)
    return vis


movenet = None
try:
    from movenet.movenet_trt import MoveNetTRT, draw_keypoints as draw_keypoints_trt
    print('[INFO] Loading MoveNet TRT engine: {}'.format(ENGINE_PATH))
    movenet = MoveNetTRT(ENGINE_PATH)
    draw_keypoints = draw_keypoints_trt
    print('[INFO] MoveNet backend: TensorRT')
except Exception as e:
    print('[WARN] TensorRT unavailable, falling back to PyTorch MoveNet:', e)
    from build_dataset import MoveNetTorch
    movenet_dir = os.path.join(ROOT, 'movenet')
    weights_rel = os.path.join('output', 'movenet.pth')
    weights_abs = os.path.join(movenet_dir, weights_rel)
    if not os.path.isfile(weights_abs):
        raise FileNotFoundError('Missing MoveNet .pth: {}'.format(weights_abs))
    cwd = os.getcwd()
    os.chdir(movenet_dir)
    try:
        movenet = MoveNetTorch(weights_rel)
    finally:
        os.chdir(cwd)
    draw_keypoints = draw_keypoints_fallback
    print('[INFO] MoveNet backend: PyTorch')

print('[INFO] Loading GRU: {}'.format(GRU_PATH))
gru, ckpt = FallDetectionGRU.load_from(GRU_PATH, map_location='cpu')
gru.eval()
SEQ_LEN = ckpt.get('seq_len', 30)
print('     seq_len    : {}'.format(SEQ_LEN))
print('     params     : {:,}'.format(gru.num_parameters()))
if PROB_THRESHOLD is None:
    PROB_THRESHOLD = float(ckpt.get('best_threshold', 0.6))
print('     prob_thr   : {:.2f}'.format(PROB_THRESHOLD))
if 'val_metrics' in ckpt:
    m = ckpt['val_metrics']
    print('     val F1     : {:.3f}  (acc={:.3f}, P={:.3f}, R={:.3f})'.format(
        m.get('f1', 0), m.get('acc', 0), m.get('precision', 0), m.get('recall', 0)))
if 'best_threshold_metrics' in ckpt:
    bm = ckpt['best_threshold_metrics']
    print('     val F1@thr : {:.3f}  (P={:.3f}, R={:.3f})'.format(
        bm.get('f1', 0), bm.get('precision', 0), bm.get('recall', 0)))

# Auto-detect feature dim from loaded model so old (51-dim) and new (54-dim) models both work
USE_EXTRA_FEATURES = (gru.input_dim == 54)
print("     extra_feat : {}".format(USE_EXTRA_FEATURES))


[WARN] TensorRT unavailable, falling back to PyTorch MoveNet: No module named 'tensorrt'


C:\Users\KLssis\AppData\Roaming\Python\Python313\site-packages\torch\nn\_reduction.py:51: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  warnings.warn(warning.format(ret))


--------------------------------------------------
{'GPU_ID': '0', 'num_workers': 8, 'random_seed': 42, 'cfg_verbose': True, 'save_dir': 'output/', 'num_classes': 17, 'width_mult': 1.0, 'img_size': 192, 'img_path': './data/cropped/imgs', 'train_label_path': './data/cropped/train2017.json', 'val_label_path': './data/cropped/val2017.json', 'balance_data': False, 'log_interval': 10, 'save_best_only': True, 'pin_memory': True, 'learning_rate': 0.001, 'batch_size': 64, 'epochs': 120, 'optimizer': 'Adam', 'scheduler': 'MultiStepLR-70,100-0.1', 'weight_decay': 0.0005, 'class_weight': None, 'clip_gradient': 5, 'test_img_path': './data/test_imgs', 'exam_label_path': '../data/all/data_all_new.json', 'eval_img_path': '../data/eval/imgs', 'eval_label_path': '../data/eval/mypc.json'}
--------------------------------------------------
[OK] Weights loaded cleanly: output\movenet.pth
[OK] MoveNet PyTorch loaded
     device : cuda
     input  : (1, 3, 192, 192)
[INFO] MoveNet backend: PyTorch
[INFO] Lo

In [3]:
# === UI overlay + rule fusion + run loop helpers =======================
KP_NOSE = 0
KP_L_SHOULDER = 5
KP_R_SHOULDER = 6
KP_L_HIP = 11
KP_R_HIP = 12


def _center_from_kp(kp, idx_a, idx_b, score_thr):
    pa = kp[idx_a]
    pb = kp[idx_b]
    a_ok = pa[2] >= score_thr
    b_ok = pb[2] >= score_thr
    if a_ok and b_ok:
        return (pa[:2] + pb[:2]) * 0.5
    if a_ok:
        return pa[:2]
    if b_ok:
        return pb[:2]
    return None


def _init_rule_state():
    return {
        'hip_hist':   collections.deque(maxlen=RULE_DROP_WINDOW + 1),
        'scale_hist': collections.deque(maxlen=RULE_DROP_WINDOW + 1),
        'speed_hist': collections.deque(maxlen=RULE_STILL_WINDOW),
        'angle_hist': collections.deque(maxlen=RULE_SUSTAINED_ANGLE_WINDOW),
        'drop_timer': 0,
        'prev_hip':   None,
    }


def _update_rule(kp, state):
    if not RULE_ENABLE:
        return False, {}

    hip      = _center_from_kp(kp, KP_L_HIP,      KP_R_HIP,      RULE_KP_SCORE_THR)
    shoulder = _center_from_kp(kp, KP_L_SHOULDER,  KP_R_SHOULDER, RULE_KP_SCORE_THR)
    if hip is None or shoulder is None:
        state['drop_timer'] = max(state['drop_timer'] - 1, 0)
        state['prev_hip'] = None
        return False, {'reason': 'kp_missing'}

    scale = float(np.linalg.norm(shoulder - hip))
    if scale < RULE_MIN_SCALE:
        state['drop_timer'] = max(state['drop_timer'] - 1, 0)
        state['prev_hip'] = hip
        return False, {'reason': 'scale_small'}

    vec   = shoulder - hip
    angle = float(np.degrees(np.arctan2(abs(vec[0]), abs(vec[1]) + 1e-6)))

    state['hip_hist'].append(hip)
    state['scale_hist'].append(scale)
    state['angle_hist'].append(angle)

    # Rapid-drop event
    drop_norm = None
    if len(state['hip_hist']) >= RULE_DROP_WINDOW + 1:
        dy        = float(state['hip_hist'][-1][1] - state['hip_hist'][0][1])
        scale_ref = max(float(np.mean(state['scale_hist'])), 1.0)
        drop_norm = dy / scale_ref

    drop_event  = (drop_norm is not None) and (drop_norm >= RULE_DROP_THR)
    angle_event = angle >= RULE_ANGLE_THR

    # Stillness
    speed = 0.0
    if state['prev_hip'] is not None:
        speed = float(np.linalg.norm(hip - state['prev_hip'])) / max(scale, 1.0)
    state['prev_hip'] = hip
    state['speed_hist'].append(speed)

    still_event = (len(state['speed_hist']) >= RULE_STILL_WINDOW and
                   float(np.mean(state['speed_hist'])) <= RULE_STILL_SPEED)

    # Sustained-horizontal: mean angle over the full look-back window >= threshold
    sustained_angle_event = (
        len(state['angle_hist']) >= RULE_SUSTAINED_ANGLE_WINDOW and
        float(np.mean(state['angle_hist'])) >= RULE_SUSTAINED_ANGLE_THR
    )

    # Update drop timer
    if drop_event and angle_event:
        state['drop_timer'] = RULE_DROP_HOLD
    else:
        state['drop_timer'] = max(state['drop_timer'] - 1, 0)

    # Combine rules
    rule_flag = False
    # 1. Rapid drop AND currently horizontal
    if drop_event and angle_event:
        rule_flag = True
    # 2. Recent drop (timer alive) AND still AND currently horizontal
    #    BUG FIX: old code omitted angle_event, causing FPs when someone
    #    bent over briefly then stopped later while upright.
    if state['drop_timer'] > 0 and still_event and angle_event:
        rule_flag = True
    # 3. Body sustained horizontal AND still (catches slow/silent falls)
    if sustained_angle_event and still_event:
        rule_flag = True

    info = {
        'angle':      angle,
        'drop_norm':  drop_norm if drop_norm is not None else 0.0,
        'still':      still_event,
        'sustained':  sustained_angle_event,
        'drop_timer': state['drop_timer'],
    }
    return rule_flag, info


def draw_overlay(frame, fall_prob, fused_prob, rule_active,
                 alarm_active, fps, seq_filled, seq_len):
    """Top alarm banner, bottom fall_prob bar, FPS, buffer status."""
    h, w = frame.shape[:2]

    if alarm_active:
        cv2.rectangle(frame, (0, 0), (w, 50), (0, 0, 200), -1)
        cv2.putText(frame, "FALL DETECTED", (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

    bar_x, bar_y = 10, h - 30
    bar_w, bar_h = 220, 18
    cv2.rectangle(frame, (bar_x, bar_y),
                  (bar_x + bar_w, bar_y + bar_h), (50, 50, 50), -1)
    fill      = int(bar_w * float(fused_prob))
    bar_color = (0, 165, 255) if fused_prob < PROB_THRESHOLD else (0, 0, 255)
    cv2.rectangle(frame, (bar_x, bar_y),
                  (bar_x + fill, bar_y + bar_h), bar_color, -1)
    cv2.putText(frame, "fall {:.2f} (gru {:.2f})".format(fused_prob, fall_prob),
                (bar_x + 5, bar_y + 14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    if rule_active:
        cv2.putText(frame, "RULE", (10, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.putText(frame, "{:.1f} FPS".format(fps), (w - 110, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(frame, "buf {}/{}".format(seq_filled, seq_len),
                (w - 110, h - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    return frame


def run_loop(cap, save_out=None, display=True, log_every=50):
    """Main loop: read frame -> MoveNet -> features -> GRU -> alarm -> display."""
    in_w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    in_h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    in_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    print("source: {}x{} @ {:.1f} fps".format(in_w, in_h, in_fps))

    writer = None
    if save_out:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(save_out, fourcc, in_fps, (in_w, in_h))
        print("recording to: {}".format(save_out))

    feat_ext    = KeypointFeatureExtractor(smooth_alpha=SMOOTH_ALPHA, extra_features=USE_EXTRA_FEATURES)
    feat_ext.reset()
    feat_buffer = collections.deque(maxlen=SEQ_LEN)

    rule_state = _init_rule_state()

    over_thr_count  = 0
    alarm_remaining = 0
    fall_prob       = 0.0
    fused_prob      = 0.0
    frame_idx       = 0

    t_last     = time.time()
    fps_smooth = 0.0
    alpha_fps  = 0.1

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("[INFO] stream ended at frame {}".format(frame_idx))
                break
            frame_idx += 1

            kp   = movenet.infer(frame)
            feat = feat_ext.extract(kp)
            feat_buffer.append(feat)

            if (len(feat_buffer) == SEQ_LEN and
                    frame_idx % INFERENCE_STRIDE == 0):
                seq = np.stack(list(feat_buffer), axis=0)
                with torch.no_grad():
                    x      = torch.from_numpy(seq).float().unsqueeze(0)
                    logits = gru(x)
                    probs  = torch.softmax(logits, dim=1)[0].numpy()
                fall_prob = float(probs[1])

            rule_active, _ = _update_rule(kp, rule_state)

            fused_prob = fall_prob
            if RULE_ENABLE and rule_active:
                if RULE_FORCE_ALARM or fall_prob >= GRU_MIN_FOR_RULE:
                    fused_prob = max(fall_prob, RULE_BONUS_PROB)

            if fused_prob >= PROB_THRESHOLD:
                over_thr_count += 1
            else:
                over_thr_count = 0
            if over_thr_count >= ALARM_FRAMES:
                alarm_remaining = ALARM_HOLD
                over_thr_count  = 0

            if alarm_remaining > 0:
                alarm_remaining -= 1
            alarm_active = alarm_remaining > 0

            vis = draw_keypoints(frame, kp, conf_thr=DRAW_KP_THR)

            t_now = time.time()
            dt    = max(t_now - t_last, 1e-6)
            cur_fps = 1.0 / dt
            t_last  = t_now
            fps_smooth = ((1 - alpha_fps) * fps_smooth + alpha_fps * cur_fps
                          if fps_smooth > 0 else cur_fps)

            vis = draw_overlay(vis, fall_prob, fused_prob, rule_active,
                               alarm_active, fps_smooth,
                               len(feat_buffer), SEQ_LEN)

            if writer is not None:
                writer.write(vis)

            if display:
                cv2.imshow("Fall Detection", vis)
                key = cv2.waitKey(1) & 0xFF
                if key == ord("q"):
                    print("[INFO] quit by user")
                    break

            if frame_idx % log_every == 0:
                print("[frame {:5d}] gru={:.2f} fused={:.2f} rule={} alarm={} fps={:.1f}".format(
                    frame_idx, fall_prob, fused_prob, rule_active, alarm_active, fps_smooth))

    except KeyboardInterrupt:
        print("[INFO] interrupted")
    finally:
        cap.release()
        if writer is not None:
            writer.release()
        if display:
            cv2.destroyAllWindows()
        print("[INFO] done")

print('helpers defined')


helpers defined


In [4]:
# === Run on keypoint .npz (no video) ==================================
import glob

# Set to a specific .npz if desired. If None, use the first found.
NPZ_PATH = None

if NPZ_PATH is None:
    cand = sorted(glob.glob('datasets/raw_keypoints/**/*.npz', recursive=True))
    if not cand:
        raise FileNotFoundError('No .npz found under datasets/raw_keypoints')
    NPZ_PATH = cand[0]

print('using npz:', NPZ_PATH)
npz = np.load(NPZ_PATH, allow_pickle=True)
kp_seq = npz['keypoints'].astype(np.float32)
start = int(npz['start'])
end = int(npz['end'])
malformed = bool(npz['malformed'])
video_id = str(npz['video_id']) if 'video_id' in npz else os.path.basename(NPZ_PATH)

# Build frame labels (same rule as build_dataset.py stage B)
post_fall_frames = 50
T = kp_seq.shape[0]
labels = np.zeros(T, dtype=np.int64)
if (start > 0 and end > 0) and not malformed:
    lo = max(start, 0)   # pre-fall: include falling motion (consistent with v5 training)
    hi = min(end + post_fall_frames, T - 1)
    labels[lo:hi + 1] = 1

# Feature extraction
feat_ext = KeypointFeatureExtractor(smooth_alpha=SMOOTH_ALPHA, extra_features=USE_EXTRA_FEATURES)
feat_seq = feat_ext.extract_sequence(kp_seq)

# Sliding-window inference (per-frame)
rule_state = _init_rule_state()
fall_prob = np.zeros(T, dtype=np.float32)
fused_prob = np.zeros(T, dtype=np.float32)
rule_flag = np.zeros(T, dtype=np.bool_)

for t in range(T):
    rule_active, _ = _update_rule(kp_seq[t], rule_state)
    rule_flag[t] = rule_active

    if t >= SEQ_LEN - 1:
        window = feat_seq[t - SEQ_LEN + 1:t + 1]
        with torch.no_grad():
            x = torch.from_numpy(window).float().unsqueeze(0)
            logits = gru(x)
            probs = torch.softmax(logits, dim=1)[0].numpy()
        fall_prob[t] = float(probs[1])
    else:
        fall_prob[t] = 0.0

    fused = fall_prob[t]
    if RULE_ENABLE and rule_active:
        if RULE_FORCE_ALARM or fall_prob[t] >= GRU_MIN_FOR_RULE:
            fused = max(fused, RULE_BONUS_PROB)
    fused_prob[t] = fused

# Frame-level metrics
pred = fused_prob >= PROB_THRESHOLD

tp = int(((pred == 1) & (labels == 1)).sum())
fp = int(((pred == 1) & (labels == 0)).sum())
fn = int(((pred == 0) & (labels == 1)).sum())
precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-9)

print('video:', video_id)
print('frames:', T, 'fall_frames:', int(labels.sum()), 'malformed:', malformed)
print('thr: {:.2f}  tp={} fp={} fn={}'.format(PROB_THRESHOLD, tp, fp, fn))
print('precision={:.3f} recall={:.3f} f1={:.3f}'.format(precision, recall, f1))
print('max_gru_prob={:.3f} max_fused_prob={:.3f} rule_hits={}'.format(
    float(fall_prob.max()), float(fused_prob.max()), int(rule_flag.sum())))

using npz: datasets/raw_keypoints\Coffee_room_01\video (1).npz
video: Coffee_room_01_video_(1)
frames: 157 fall_frames: 83 malformed: False
thr: 0.75  tp=83 fp=27 fn=0
precision=0.755 recall=1.000 f1=0.860
max_gru_prob=0.996 max_fused_prob=0.996 rule_hits=0


In [5]:
# === Batch tuning on npz sample =======================================
import random

SEED             = 42
NUM_SAMPLES      = 50
N_TRIALS         = 40   # was 6; more trials = better search coverage
POST_FALL_FRAMES = 50

all_npz_raw = sorted(glob.glob('datasets/raw_keypoints/**/*.npz', recursive=True))
if not all_npz_raw:
    raise FileNotFoundError('No .npz found under datasets/raw_keypoints')

# Only evaluate on ORIGINAL (non-augmented) files.
# Augmented files (_flip/_fast/_slow/_rev) are TRAINING DATA — including
# them in the test pool would inflate metrics via data leakage.
_AUG_SUFFIXES = ('_flip.npz', '_fast.npz', '_slow.npz', '_rev.npz')
all_npz = [p for p in all_npz_raw if not any(p.endswith(s) for s in _AUG_SUFFIXES)]
print(f'npz pool: {len(all_npz_raw)} total → {len(all_npz)} original-only (excluded {len(all_npz_raw)-len(all_npz)} augmented training files)')

rng = random.Random(SEED)
rng.shuffle(all_npz)
npz_list = all_npz[:min(NUM_SAMPLES, len(all_npz))]
print('npz sample:', len(npz_list))


def build_labels_from_npz(npz_obj, post_fall_frames):
    start     = int(npz_obj['start'])
    end       = int(npz_obj['end'])
    malformed = bool(npz_obj['malformed'])
    T         = int(npz_obj['keypoints'].shape[0])
    labels    = np.zeros(T, dtype=np.int64)
    if (start > 0 and end > 0) and not malformed:
        lo = max(start, 0)   # pre-fall: include falling motion (consistent with v5 training)
        hi = min(end + post_fall_frames, T - 1)
        labels[lo:hi + 1] = 1
    return labels


def compute_gru_probs(kp_seq):
    feat_ext = KeypointFeatureExtractor(smooth_alpha=SMOOTH_ALPHA, extra_features=USE_EXTRA_FEATURES)
    feat_seq = feat_ext.extract_sequence(kp_seq)
    T        = feat_seq.shape[0]
    probs    = np.zeros(T, dtype=np.float32)
    for t in range(SEQ_LEN - 1, T):
        window = feat_seq[t - SEQ_LEN + 1:t + 1]
        with torch.no_grad():
            x      = torch.from_numpy(window).float().unsqueeze(0)
            logits = gru(x)
            p      = torch.softmax(logits, dim=1)[0].numpy()
        probs[t] = float(p[1])
    return probs


precomp = []
for path in npz_list:
    npz_          = np.load(path, allow_pickle=True)
    kp_seq        = npz_['keypoints'].astype(np.float32)
    labels        = build_labels_from_npz(npz_, POST_FALL_FRAMES)
    fall_prob_seq = compute_gru_probs(kp_seq)
    precomp.append((path, kp_seq, labels, fall_prob_seq))

print('precompute done')


def _init_rule_state_params(params):
    return {
        'hip_hist':   collections.deque(maxlen=params['RULE_DROP_WINDOW'] + 1),
        'scale_hist': collections.deque(maxlen=params['RULE_DROP_WINDOW'] + 1),
        'speed_hist': collections.deque(maxlen=params['RULE_STILL_WINDOW']),
        'angle_hist': collections.deque(maxlen=params['RULE_SUSTAINED_ANGLE_WINDOW']),
        'drop_timer': 0,
        'prev_hip':   None,
    }


def _update_rule_params(kp, state, params):
    hip      = _center_from_kp(kp, KP_L_HIP,     KP_R_HIP,      params['RULE_KP_SCORE_THR'])
    shoulder = _center_from_kp(kp, KP_L_SHOULDER, KP_R_SHOULDER, params['RULE_KP_SCORE_THR'])
    if hip is None or shoulder is None:
        state['drop_timer'] = max(state['drop_timer'] - 1, 0)
        state['prev_hip'] = None
        return False

    scale = float(np.linalg.norm(shoulder - hip))
    if scale < params['RULE_MIN_SCALE']:
        state['drop_timer'] = max(state['drop_timer'] - 1, 0)
        state['prev_hip'] = hip
        return False

    vec   = shoulder - hip
    angle = float(np.degrees(np.arctan2(abs(vec[0]), abs(vec[1]) + 1e-6)))

    state['hip_hist'].append(hip)
    state['scale_hist'].append(scale)
    state['angle_hist'].append(angle)

    drop_norm = None
    if len(state['hip_hist']) >= params['RULE_DROP_WINDOW'] + 1:
        dy        = float(state['hip_hist'][-1][1] - state['hip_hist'][0][1])
        scale_ref = max(float(np.mean(state['scale_hist'])), 1.0)
        drop_norm = dy / scale_ref

    drop_event  = (drop_norm is not None) and (drop_norm >= params['RULE_DROP_THR'])
    angle_event = angle >= params['RULE_ANGLE_THR']

    speed = 0.0
    if state['prev_hip'] is not None:
        speed = float(np.linalg.norm(hip - state['prev_hip'])) / max(scale, 1.0)
    state['prev_hip'] = hip
    state['speed_hist'].append(speed)

    still_event = (len(state['speed_hist']) >= params['RULE_STILL_WINDOW'] and
                   float(np.mean(state['speed_hist'])) <= params['RULE_STILL_SPEED'])

    sustained_angle_event = (
        len(state['angle_hist']) >= params['RULE_SUSTAINED_ANGLE_WINDOW'] and
        float(np.mean(state['angle_hist'])) >= params['RULE_SUSTAINED_ANGLE_THR']
    )

    if drop_event and angle_event:
        state['drop_timer'] = params['RULE_DROP_HOLD']
    else:
        state['drop_timer'] = max(state['drop_timer'] - 1, 0)

    if drop_event and angle_event:
        return True
    if state['drop_timer'] > 0 and still_event and angle_event:
        return True
    if sustained_angle_event and still_event:
        return True
    return False


def fuse_probs(kp_seq, fall_prob_seq, params):
    T     = kp_seq.shape[0]
    fused = fall_prob_seq.copy()
    state = _init_rule_state_params(params)
    for t in range(T):
        rule_active = _update_rule_params(kp_seq[t], state, params)
        if rule_active:
            if params['RULE_FORCE_ALARM'] or fused[t] >= params['GRU_MIN_FOR_RULE']:
                fused[t] = max(fused[t], params['RULE_BONUS_PROB'])
    return fused


def eval_params(params, prob_thr):
    tp = fp = fn = 0
    for _, kp_seq, labels, fall_prob_seq in precomp:
        fused = fuse_probs(kp_seq, fall_prob_seq, params)
        pred  = fused >= prob_thr
        tp += int(((pred == 1) & (labels == 1)).sum())
        fp += int(((pred == 1) & (labels == 0)).sum())
        fn += int(((pred == 0) & (labels == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    f1        = 2 * precision * recall / max(precision + recall, 1e-9)
    return {'tp': tp, 'fp': fp, 'fn': fn,
            'precision': precision, 'recall': recall, 'f1': f1}


base_params = {
    'RULE_FORCE_ALARM':            RULE_FORCE_ALARM,
    'GRU_MIN_FOR_RULE':            GRU_MIN_FOR_RULE,
    'RULE_BONUS_PROB':             RULE_BONUS_PROB,
    'RULE_DROP_THR':               RULE_DROP_THR,
    'RULE_ANGLE_THR':              RULE_ANGLE_THR,
    'RULE_STILL_SPEED':            RULE_STILL_SPEED,
    'RULE_DROP_WINDOW':            RULE_DROP_WINDOW,
    'RULE_STILL_WINDOW':           RULE_STILL_WINDOW,
    'RULE_DROP_HOLD':              RULE_DROP_HOLD,
    'RULE_KP_SCORE_THR':           RULE_KP_SCORE_THR,
    'RULE_MIN_SCALE':              RULE_MIN_SCALE,
    'RULE_SUSTAINED_ANGLE_WINDOW': RULE_SUSTAINED_ANGLE_WINDOW,
    'RULE_SUSTAINED_ANGLE_THR':    RULE_SUSTAINED_ANGLE_THR,
}


def sample_params(rng_obj):
    return {
        'RULE_FORCE_ALARM':            rng_obj.choice([True, False]),
        'GRU_MIN_FOR_RULE':            rng_obj.choice([0.15, 0.20, 0.25, 0.30, 0.35]),
        'RULE_BONUS_PROB':             rng_obj.choice([0.80, 0.85, 0.90, 0.95]),
        'RULE_DROP_THR':               rng_obj.choice([0.50, 0.60, 0.65, 0.70, 0.75, 0.80]),
        'RULE_ANGLE_THR':              rng_obj.choice([40.0, 45.0, 50.0, 55.0, 60.0]),
        'RULE_STILL_SPEED':            rng_obj.choice([0.04, 0.06, 0.08, 0.10, 0.12]),
        'RULE_DROP_WINDOW':            rng_obj.choice([4, 5, 6, 8]),
        'RULE_STILL_WINDOW':           rng_obj.choice([8, 10, 12, 15, 20]),
        'RULE_DROP_HOLD':              rng_obj.choice([8, 10, 12, 15, 20]),
        'RULE_KP_SCORE_THR':           RULE_KP_SCORE_THR,
        'RULE_MIN_SCALE':              RULE_MIN_SCALE,
        'RULE_SUSTAINED_ANGLE_WINDOW': rng_obj.choice([15, 20, 25, 30]),
        'RULE_SUSTAINED_ANGLE_THR':    rng_obj.choice([55.0, 60.0, 65.0, 70.0]),
    }


best        = None
best_params = None
print('prob_thr:', PROB_THRESHOLD)

for i in range(N_TRIALS):
    params  = base_params if i == 0 else sample_params(rng)
    metrics = eval_params(params, PROB_THRESHOLD)
    print('trial {:2d}  f1={:.3f}  P={:.3f}  R={:.3f}  tp={} fp={} fn={}'.format(
        i, metrics['f1'], metrics['precision'], metrics['recall'],
        metrics['tp'], metrics['fp'], metrics['fn']))
    if best is None or metrics['f1'] > best['f1']:
        best        = metrics
        best_params = params

print('\nBEST: f1={:.3f}  P={:.3f}  R={:.3f}'.format(
    best['f1'], best['precision'], best['recall']))
print('BEST PARAMS:', best_params)

# Apply best params to globals for subsequent runs
RULE_FORCE_ALARM            = best_params['RULE_FORCE_ALARM']
GRU_MIN_FOR_RULE            = best_params['GRU_MIN_FOR_RULE']
RULE_BONUS_PROB             = best_params['RULE_BONUS_PROB']
RULE_DROP_THR               = best_params['RULE_DROP_THR']
RULE_ANGLE_THR              = best_params['RULE_ANGLE_THR']
RULE_STILL_SPEED            = best_params['RULE_STILL_SPEED']
RULE_DROP_WINDOW            = best_params['RULE_DROP_WINDOW']
RULE_STILL_WINDOW           = best_params['RULE_STILL_WINDOW']
RULE_DROP_HOLD              = best_params['RULE_DROP_HOLD']
RULE_SUSTAINED_ANGLE_WINDOW = best_params['RULE_SUSTAINED_ANGLE_WINDOW']
RULE_SUSTAINED_ANGLE_THR    = best_params['RULE_SUSTAINED_ANGLE_THR']
print('updated rule params in notebook')


npz pool: 641 total → 163 original-only (excluded 478 augmented training files)
npz sample: 50
precompute done
prob_thr: 0.75
trial  0  f1=0.796  P=0.678  R=0.963  tp=1919 fp=910 fn=73
trial  1  f1=0.755  P=0.621  R=0.963  tp=1919 fp=1172 fn=73
trial  2  f1=0.796  P=0.678  R=0.963  tp=1919 fp=910 fn=73
trial  3  f1=0.796  P=0.678  R=0.963  tp=1919 fp=910 fn=73
trial  4  f1=0.756  P=0.622  R=0.963  tp=1919 fp=1166 fn=73
trial  5  f1=0.796  P=0.678  R=0.963  tp=1919 fp=911 fn=73
trial  6  f1=0.796  P=0.678  R=0.963  tp=1919 fp=910 fn=73
trial  7  f1=0.796  P=0.678  R=0.963  tp=1919 fp=911 fn=73
trial  8  f1=0.779  P=0.654  R=0.963  tp=1919 fp=1017 fn=73
trial  9  f1=0.796  P=0.678  R=0.963  tp=1919 fp=911 fn=73
trial 10  f1=0.787  P=0.665  R=0.963  tp=1919 fp=967 fn=73
trial 11  f1=0.768  P=0.638  R=0.963  tp=1919 fp=1088 fn=73
trial 12  f1=0.772  P=0.644  R=0.963  tp=1919 fp=1061 fn=73
trial 13  f1=0.763  P=0.631  R=0.963  tp=1919 fp=1122 fn=73
trial 14  f1=0.778  P=0.652  R=0.963  tp=1

In [6]:
# === Threshold sweep (optimize F1 on sample) ===========================
thr_min = 0.10
thr_max = 0.99
thr_step = 0.01

params = best_params if 'best_params' in globals() and best_params else base_params

best_thr = None
best_metrics = None

thr_values = np.arange(thr_min, thr_max + 1e-9, thr_step)
for thr in thr_values:
    metrics = eval_params(params, float(thr))
    if best_metrics is None or metrics['f1'] > best_metrics['f1']:
        best_metrics = metrics
        best_thr = float(thr)

print('THRESHOLD BEST: thr={:.2f}  f1={:.3f}  P={:.3f}  R={:.3f}  tp={} fp={} fn={}'.format(
    best_thr, best_metrics['f1'], best_metrics['precision'], best_metrics['recall'],
    best_metrics['tp'], best_metrics['fp'], best_metrics['fn']))

# Apply threshold for subsequent runs
PROB_THRESHOLD = best_thr
print('updated PROB_THRESHOLD:', PROB_THRESHOLD)

THRESHOLD BEST: thr=0.91  f1=0.810  P=0.764  R=0.862  tp=1717 fp=530 fn=275
updated PROB_THRESHOLD: 0.9099999999999996


In [7]:
# === Larger sample re-tune + fine threshold ============================
# all_npz here refers to the original-only pool built in Cell 5
NUM_SAMPLES = min(100, len(all_npz))
N_TRIALS = 6

rng = random.Random(SEED)
rng.shuffle(all_npz)
npz_list = all_npz[:NUM_SAMPLES]

precomp = []
for path in npz_list:
    npz = np.load(path, allow_pickle=True)
    kp_seq = npz['keypoints'].astype(np.float32)
    labels = build_labels_from_npz(npz, POST_FALL_FRAMES)
    fall_prob_seq = compute_gru_probs(kp_seq)
    precomp.append((path, kp_seq, labels, fall_prob_seq))

print('precompute done for sample:', len(precomp))

# baseline with current params
params = best_params if 'best_params' in globals() and best_params else base_params
base_metrics = eval_params(params, PROB_THRESHOLD)
print('baseline: f1={:.3f} P={:.3f} R={:.3f} tp={} fp={} fn={}'.format(
    base_metrics['f1'], base_metrics['precision'], base_metrics['recall'],
    base_metrics['tp'], base_metrics['fp'], base_metrics['fn']))

best = None
best_params_local = None

for i in range(N_TRIALS):
    params = base_params if i == 0 else sample_params(rng)
    metrics = eval_params(params, PROB_THRESHOLD)
    print('trial {}  f1={:.3f}  P={:.3f}  R={:.3f}  tp={} fp={} fn={}'.format(
        i, metrics['f1'], metrics['precision'], metrics['recall'],
        metrics['tp'], metrics['fp'], metrics['fn']))
    if best is None or metrics['f1'] > best['f1']:
        best = metrics
        best_params_local = params

print('\nBEST (sample {}): f1={:.3f}  P={:.3f}  R={:.3f}'.format(
    len(precomp), best['f1'], best['precision'], best['recall']))
print('BEST PARAMS:', best_params_local)

# Apply best params to globals
best_params = best_params_local
RULE_FORCE_ALARM = best_params['RULE_FORCE_ALARM']
GRU_MIN_FOR_RULE = best_params['GRU_MIN_FOR_RULE']
RULE_BONUS_PROB = best_params['RULE_BONUS_PROB']
RULE_DROP_THR = best_params['RULE_DROP_THR']
RULE_ANGLE_THR = best_params['RULE_ANGLE_THR']
RULE_STILL_SPEED = best_params['RULE_STILL_SPEED']

# Fine threshold sweep on this sample
thr_min = 0.80
thr_max = 0.99
thr_step = 0.01

best_thr = None
best_metrics = None

thr_values = np.arange(thr_min, thr_max + 1e-9, thr_step)
for thr in thr_values:
    metrics = eval_params(best_params, float(thr))
    if best_metrics is None or metrics['f1'] > best_metrics['f1']:
        best_metrics = metrics
        best_thr = float(thr)

print('FINE THR BEST: thr={:.2f}  f1={:.3f}  P={:.3f}  R={:.3f}  tp={} fp={} fn={}'.format(
    best_thr, best_metrics['f1'], best_metrics['precision'], best_metrics['recall'],
    best_metrics['tp'], best_metrics['fp'], best_metrics['fn']))

PROB_THRESHOLD = best_thr
print('updated PROB_THRESHOLD:', PROB_THRESHOLD)

precompute done for sample: 100
baseline: f1=0.810 P=0.771 R=0.853 tp=4047 fp=1205 fn=695
trial 0  f1=0.810  P=0.771  R=0.853  tp=4047 fp=1205 fn=695
trial 1  f1=0.810  P=0.771  R=0.853  tp=4047 fp=1205 fn=695
trial 2  f1=0.810  P=0.771  R=0.853  tp=4047 fp=1205 fn=695
trial 3  f1=0.810  P=0.771  R=0.853  tp=4047 fp=1205 fn=695
trial 4  f1=0.803  P=0.759  R=0.854  tp=4049 fp=1288 fn=693
trial 5  f1=0.810  P=0.771  R=0.853  tp=4047 fp=1205 fn=695

BEST (sample 100): f1=0.810  P=0.771  R=0.853
BEST PARAMS: {'RULE_FORCE_ALARM': False, 'GRU_MIN_FOR_RULE': 0.6, 'RULE_BONUS_PROB': 0.97, 'RULE_DROP_THR': 0.75, 'RULE_ANGLE_THR': 45.0, 'RULE_STILL_SPEED': 0.06, 'RULE_DROP_WINDOW': 5, 'RULE_STILL_WINDOW': 12, 'RULE_DROP_HOLD': 12, 'RULE_KP_SCORE_THR': 0.2, 'RULE_MIN_SCALE': 20.0, 'RULE_SUSTAINED_ANGLE_WINDOW': 20, 'RULE_SUSTAINED_ANGLE_THR': 60.0}
FINE THR BEST: thr=0.90  f1=0.818  P=0.755  R=0.892  tp=4229 fp=1371 fn=513
updated PROB_THRESHOLD: 0.9000000000000001


## Run on live webcam

In [8]:
# === Live webcam ======================================================
cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  CAM_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_HEIGHT)
if not cap.isOpened():
    raise RuntimeError('could not open camera index {}'.format(CAMERA_INDEX))

run_loop(cap, save_out=SAVE_OUT, display=DISPLAY)

source: 640x480 @ 25.0 fps


error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1295: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvDestroyAllWindows'


## Run on a pre-recorded video file

In [ ]:
# === Video file =======================================================
print('opening:', VIDEO_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError('could not open video: {}'.format(VIDEO_PATH))

run_loop(cap, save_out=SAVE_OUT, display=DISPLAY)

opening: data_Le2i/Coffee_room_01/Coffee_room_01/Videos/video (1).avi


RuntimeError: could not open video: data_Le2i/Coffee_room_01/Coffee_room_01/Videos/video (1).avi

## MoveNet-only sanity check (no GRU)

Process a single frame from the configured video. Expected: `score_max >= 0.6`.
If you see `score_max <= 0.4`, preprocessing is broken (letterbox / `/255` snuck back in).

In [ ]:
# === MoveNet sanity ==================================================
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
cap.release()
assert ret, 'failed to read a frame from {}'.format(VIDEO_PATH)

kp = movenet.infer(frame)
print('frame:', frame.shape)
print('kp shape:', kp.shape)
print('score range: [{:.3f}, {:.3f}]  mean={:.3f}'.format(
    kp[:, 2].min(), kp[:, 2].max(), kp[:, 2].mean()))

vis = draw_keypoints(frame, kp, conf_thr=0.1)
cv2.imwrite('debug_pipeline_sanity.jpg', vis)
print('saved debug_pipeline_sanity.jpg')

AssertionError: failed to read a frame from data_Le2i/Coffee_room_01/Coffee_room_01/Videos/video (1).avi